In [14]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random
import os

In [10]:
load_dotenv()

True

In [11]:
google_api_key = os.getenv("GOOGLE_API_KEY")

In [16]:
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [18]:
search_tool = DuckDuckGoSearchRun(region="us-en")

@tool
def Calculator(first_num: float, Second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers. Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + Second_num
        elif operation == "sub":
            result = first_num - Second_num
        elif operation == "mul":
            result = first_num * Second_num
        elif operation == "div":
            if Second_num == 0:
                return {'error': 'Division by zero is not allowed'}
            result = first_num / Second_num
        else:
            {'error': f"Unsupported operation '{operation}'"}
            
        return {'first_num':first_num, 'Second_num':Second_num, 'operation':operation, 'result':result}
    
    except Exception as e:
        return {'error': str(e)}
    
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g., 'AAPL', 'TSLA')
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=0EUE9XC9UXA05RF5"
    r = requests.get(url)
    return r.json()

In [19]:
tools = [get_stock_price, search_tool, Calculator]

llm_with_tools = llm.bind_tools(tools) 

In [20]:
class ChatState(TypedDict):
    
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}